In [ ]:
import os

os.environ['KAGGLE_TOKEN'] = 'KGAT_b05f65b8091d47dc74e0dab6b51381e9'

!pip install kaggle -q
!kaggle datasets download -d mohamedmustafa/real-life-violence-situations-dataset -p /content/violence --unzip


Dataset URL: https://www.kaggle.com/datasets/mohamedmustafa/real-life-violence-situations-dataset
License(s): copyright-authors
100% 3.58G/3.58G [00:38<00:00, 99.1MB/s]



In [ ]:
from pathlib import Path

violence_dir = Path('/content/violence')
for item in sorted(violence_dir.rglob('*')):
    level = len(item.relative_to(violence_dir).parts)
    if level <= 2:
        count = len(list(item.iterdir())) if item.is_dir() else ''
        print('  ' * (level-1) + item.name + (f'/ → {count} items' if item.is_dir() else ''))


Real Life Violence Dataset/ → 2 items
  NonViolence/ → 1000 items
  Violence/ → 1000 items
real life violence situations/ → 1 items
  Real Life Violence Dataset/ → 2 items


In [ ]:
import cv2, random
from pathlib import Path

FRAMES_POR_VIDEO = 10
BASE = Path('/content/violence/Real Life Violence Dataset')

for clase, label in [('Violence', 'Fight'), ('NonViolence', 'NonFight')]:
    src = BASE / clase
    videos = list(src.glob('*.mp4')) + list(src.glob('*.avi'))

    # 80% train, 20% val
    random.shuffle(videos)
    split_idx = int(len(videos) * 0.8)
    splits = [('train', videos[:split_idx]), ('val', videos[split_idx:])]

    for split_name, split_videos in splits:
        dst = Path(f'/content/frames/{split_name}/{label}')
        dst.mkdir(parents=True, exist_ok=True)
        print(f"{split_name}/{label}: {len(split_videos)} videos")

        for video in split_videos:
            cap = cv2.VideoCapture(str(video))
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            indices = sorted(random.sample(range(max(total,1)), min(FRAMES_POR_VIDEO, total)))
            for i, idx in enumerate(indices):
                cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                ret, frame = cap.read()
                if ret:
                    cv2.imwrite(str(dst / f'{video.stem}_{i}.jpg'), frame)
            cap.release()

print("✅ Frames extraídos")


train/Fight: 800 videos
val/Fight: 200 videos
train/NonFight: 800 videos
val/NonFight: 200 videos
✅ Frames extraídos


In [ ]:
import torch
from torchvision import datasets, transforms, models
import torch.nn as nn

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds = datasets.ImageFolder('/content/frames/train', transform=transform)
val_ds   = datasets.ImageFolder('/content/frames/val',   transform=transform)
print("Clases:", train_ds.classes)  # ['Fight', 'NonFight']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Dispositivo:", device)

modelo = models.resnet18(pretrained=True)
modelo.fc = nn.Linear(modelo.fc.in_features, 2)
modelo = modelo.to(device)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(modelo.parameters(), lr=0.0001)

for epoch in range(10):
    modelo.train()
    loss_total = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(modelo(imgs), labels)
        loss.backward(); optimizer.step()
        loss_total += loss.item()
    print(f"Época {epoch+1}/10 — Loss: {loss_total/len(train_loader):.4f}")


Clases: ['Fight', 'NonFight']
Dispositivo: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 44.7M/44.7M [00:00<00:00, 180MB/s]


Época 1/10 — Loss: 0.0973
Época 2/10 — Loss: 0.0270
Época 3/10 — Loss: 0.0213
Época 4/10 — Loss: 0.0150
Época 5/10 — Loss: 0.0151
Época 6/10 — Loss: 0.0082
Época 7/10 — Loss: 0.0163
Época 8/10 — Loss: 0.0095
Época 9/10 — Loss: 0.0070
Época 10/10 — Loss: 0.0073


In [ ]:
# ── Evaluación en validación ──────────────────────────────────────────────────
import torch
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_ds_eval = datasets.ImageFolder('/content/frames/val', transform=transform_val)
val_loader  = torch.utils.data.DataLoader(val_ds_eval, batch_size=32, shuffle=False, num_workers=2)

modelo.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = modelo(imgs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

clases = val_ds_eval.classes  # ['Fight', 'NonFight']
acc = accuracy_score(all_labels, all_preds)
print(f"
☁ METRICAS DE EVALUACION - Conjunto de Validacion")
print(f"{'='*50}")
print(f"Accuracy total: {acc*100:.2f}%")
print(f"
{classification_report(all_labels, all_preds, target_names=clases)}")

cm = confusion_matrix(all_labels, all_preds)
print("Matriz de Confusion:")
print(f"              Pred Fight  Pred NonFight")
for i, clase in enumerate(clases):
    print(f"Real {clase:<10}  {cm[i][0]:^10}  {cm[i][1]:^13}")


In [ ]:
# ── Grafica de perdida por epoca ──────────────────────────────────────────────
import matplotlib.pyplot as plt

losses = [0.0973, 0.0270, 0.0213, 0.0150, 0.0151, 0.0082, 0.0163, 0.0095, 0.0070, 0.0073]
plt.figure(figsize=(8,4))
plt.plot(range(1, 11), losses, marker='o', color='steelblue', linewidth=2)
plt.title('Perdida de Entrenamiento por Epoca - Modelo Pelea (ResNet-18)')
plt.xlabel('Epoca')
plt.ylabel('Loss')
plt.xticks(range(1,11))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('curva_perdida_pelea.png', dpi=120)
plt.show()
print("Grafica guardada")


In [ ]:
import shutil
torch.save(modelo.state_dict(), 'modelo_pelea.pth')
shutil.copy('modelo_pelea.pth', '/content/drive/MyDrive/modelo_pelea.pth')
print("✅ Guardado en Drive")


✅ Guardado en Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import shutil
shutil.copy('/content/drive/MyDrive/modelo_pelea.pth', '/content/modelo_pelea.pth')
print("✅")


✅


In [ ]:
from google.colab import files
files.download('modelo_pelea.pth')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
print(os.path.exists('/content/violence/Real Life Violence Dataset'))


False
